In [ ]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu pypdf

In [ ]:
"""
=============================================================================
FINAL MILESTONE: 3D U-NET INFERENCE + RAG TEXTBOOK SEARCH (PRODUCTION)
=============================================================================
"""
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ==========================================
# 1. DEFINE THE 3D U-NET ARCHITECTURE
# ==========================================
class DoubleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class LightweightUNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=3):
        super().__init__()
        self.inc = DoubleConv3D(in_channels, 16)
        self.down1 = nn.Sequential(nn.MaxPool3d(2), DoubleConv3D(16, 32))
        self.down2 = nn.Sequential(nn.MaxPool3d(2), DoubleConv3D(32, 64))
        self.down3 = nn.Sequential(nn.MaxPool3d(2), DoubleConv3D(64, 128))
        self.up1 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv3D(128, 64) 
        self.up2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv3D(64, 32)
        self.up3 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv3D(32, 16)
        self.outc = nn.Conv3d(16, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up1(x4)
        x = torch.cat([x3, x], dim=1) 
        x = self.conv_up1(x)
        x = self.up2(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv_up2(x)
        x = self.up3(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv_up3(x)
        return self.outc(x)

# ==========================================
# 2. BUILD THE TEXTBOOK DATABASE (RAG)
# ==========================================
def build_book_database(pdf_directory):
    print("Loading Medical Textbooks into FAISS Database...")
    documents = []
    if os.path.exists(pdf_directory):
        for file in os.listdir(pdf_directory):
            if file.endswith('.pdf'):
                loader = PyPDFLoader(os.path.join(pdf_directory, file))
                documents.extend(loader.load())
    else:
        print(f"Warning: Textbook directory not found at {pdf_directory}")
        
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    book_chunks = text_splitter.split_documents(documents)
    
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_database = FAISS.from_documents(book_chunks, embeddings)
    print("Database complete! Textbooks are ready for search.")
    return vector_database

book_db = build_book_database('/kaggle/input/datasets/hafzanoor/neuro-oncology-books-collection')

# ==========================================
# 3. LOAD THE TRAINED AI WEIGHTS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LightweightUNet3D(in_channels=4, out_channels=3).to(device)
weights_path = '/kaggle/input/notebooks/hafzanoor/milestone2/best_unet_model.pth' 

if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    print("Real MPhil AI Weights Loaded Successfully!")
else:
    print(f"ERROR: Weights not found at {weights_path}.")

# ==========================================
# 3b. PREPROCESSING (must match training)
# ==========================================
def normalize_intensity(volume):
    """Per-channel z-score on brain voxels (> 0), background stays 0. volume: (C, H, W, D)."""
    out = torch.zeros_like(volume)
    for ch in range(volume.shape[0]):
        v = volume[ch]
        brain = v > 0
        if brain.any():
            m, sd = v[brain].mean(), v[brain].std()
            out[ch][brain] = (v[brain] - m) / sd if sd > 0 else v[brain]
    return out

REGION_TEXT = {0: "peritumoral edema and whole tumor extent", 1: "tumor core with necrosis",
               2: "contrast-enhancing tumor"}

# ==========================================
# 4. DEFINE THE VISUAL PIPELINE
# ==========================================
def process_mri_with_textbooks(mri_file):
    if mri_file is None:
        return None, "⚠️ Error: No file uploaded.", "Please upload a 3D MRI volume."
        
    if not mri_file.name.lower().endswith(('.nii', '.nii.gz')):
        return None, "❌ Fatal Error: Volumetric .nii/.nii.gz required.", "Inference aborted."
        
    try:
        # Load and permute NIfTI
        img = nib.load(mri_file.name)
        img_data = img.get_fdata()
        raw_tensor = torch.tensor(img_data, dtype=torch.float32)
        
        if len(raw_tensor.shape) == 4 and raw_tensor.shape[3] == 4:
            reordered_tensor = raw_tensor.permute(3, 0, 1, 2)
        elif len(raw_tensor.shape) == 4 and raw_tensor.shape[0] == 4:
            reordered_tensor = raw_tensor 
        else:
            return None, f"❌ Error: Expected 4-channel MRI.", "Inference aborted."
        
        input_tensor = normalize_intensity(reordered_tensor).unsqueeze(0).to(device)
        
        # Dynamic padding to a multiple of 8 (spatial dims are H, W, D)
        H, W, D = input_tensor.shape[2:]
        pad_h, pad_w, pad_d = [(8 - n % 8) % 8 for n in (H, W, D)]
        input_tensor = F.pad(input_tensor, (0, pad_d, 0, pad_w, 0, pad_h))
        
        # Inference
        with torch.no_grad():
            output = model(input_tensor)
            output = output[:, :, :H, :W, :D]
            prediction = (torch.sigmoid(output) > 0.5).float()
            
        # Mathematical Metrics (3 Classes)
        # Class 0: Whole Tumor, Class 1: Tumor Core, Class 2: Enhancing Tumor
        voxel_mm3 = float(np.prod(img.header.get_zooms()[:3]))
        vol_wt = prediction[0, 0].sum().item() * voxel_mm3
        vol_tc = prediction[0, 1].sum().item() * voxel_mm3
        vol_et = prediction[0, 2].sum().item() * voxel_mm3
        
        status_text = (
            f"✅ Segmentation Complete\n"
            f"Whole Tumor: {vol_wt:,.0f} mm³\n"
            f"Tumor Core: {vol_tc:,.0f} mm³\n"
            f"Enhancing Tumor: {vol_et:,.0f} mm³"
        )
        
        # Visualization: axial slice = middle of the last (depth) axis
        mid_slice = D // 2
        
        # Extract 2D arrays for the base MRI and the 3 masks
        base_mri = reordered_tensor[0, :, :, mid_slice].cpu().numpy().T  # FLAIR channel
        mask_wt = prediction[0, 0, :, :, mid_slice].cpu().numpy().T
        mask_tc = prediction[0, 1, :, :, mid_slice].cpu().numpy().T
        mask_et = prediction[0, 2, :, :, mid_slice].cpu().numpy().T
        
        # Generate Matplotlib Figure
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(base_mri, cmap='gray', origin='lower')  # Base brain
        
        # Overlay masks (transparent where 0)
        ax.imshow(np.ma.masked_where(mask_wt == 0, mask_wt), cmap='Reds', alpha=0.4)
        ax.imshow(np.ma.masked_where(mask_tc == 0, mask_tc), cmap='Greens', alpha=0.6)
        ax.imshow(np.ma.masked_where(mask_et == 0, mask_et), cmap='Blues', alpha=0.8)
        
        ax.axis('off')
        ax.set_title(f"Axial Slice {mid_slice} Segmentation")
        
        # Custom Legend
        red_patch = mpatches.Patch(color='red', alpha=0.4, label='Whole Tumor')
        green_patch = mpatches.Patch(color='green', alpha=0.6, label='Tumor Core')
        blue_patch = mpatches.Patch(color='blue', alpha=0.8, label='Enhancing Tumor')
        plt.legend(handles=[red_patch, green_patch, blue_patch], loc='upper right')
        
        # RAG search: describe which tumor regions were found
        found = [REGION_TEXT[k] for k, v in enumerate((vol_wt, vol_tc, vol_et)) if v > 0]
        search_query = ("glioma on multiparametric brain MRI showing " + ", ".join(found)) if found else "normal brain MRI without tumor"
        relevant_chunks = book_db.similarity_search(search_query, k=2)
        
        clinical_reference = f"🔍 TEXTBOOK REFERENCES FOR IDENTIFIED LESION:\n\n"
        for i, chunk in enumerate(relevant_chunks):
            page_num = chunk.metadata.get('page', 'Unknown')
            clinical_reference += f"--- Reference {i+1} (Page {page_num}) ---\n{chunk.page_content}\n\n"
            
        return fig, status_text, clinical_reference
        
    except Exception as e:
        return None, f"Processing Error: {str(e)}", "Inference failed."

# ==========================================
# 5. LAUNCH THE WEB APPLICATION
# ==========================================
with gr.Blocks() as demo:
    gr.Markdown("# 🧠 3D Multimodal Brain Tumor Segmentation (Prototype) with Textbook Retrieval")
    gr.Markdown("⚠️ Research prototype for educational use only. Not for clinical diagnosis.")
    gr.Markdown("Upload a patient NIfTI (.nii.gz) file to view exact multi-class segmentation volumes, cross-sectional slices, and literature extraction.")
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload Patient .nii.gz MRI", file_types=[".nii", ".nii.gz"])
            analyze_button = gr.Button("Run Real Inference", variant="primary")
            
        with gr.Column(scale=2):
            visual_output = gr.Plot(label="Axial Slice Visualization")
            unet_output = gr.Textbox(label="Segmented tumor volumes", lines=4)
            rag_output = gr.Textbox(label="Medical Literature Extraction", lines=8)

    analyze_button.click(
        fn=process_mri_with_textbooks,
        inputs=file_input,
        outputs=[visual_output, unet_output, rag_output] 
    )

print("Thesis application ready. Launching...")
demo.launch(inline=False, share=True, theme=gr.themes.Soft())